# 05 · Modelo explicativo

**Pregunta:** ¿qué características de la zona se asocian con que los restaurantes **permanezcan** o **roten** entre 2020 y 2025? En particular: ¿los **generadores de tráfico** explican más que la población?

El modelo es **explicativo**, no predictivo: se busca entender qué se asocia con el resultado y con qué intensidad, no pronosticar restaurantes uno por uno. Por eso se usan modelos lineales generalizados, cuyos coeficientes se leen directo.

| Modelo | Unidad | Resultado | Modelo estadístico | Se lee como |
|---|---|---|---|---|
| **A. Permanencia** | Restaurante de 2020 (8,536) | ¿Sigue en 2025? (sí/no) | Logístico, errores agrupados por hexágono | Razón de momios (*odds ratio*) |
| **B. Rotación** | Hexágono con ≥ 5 restaurantes | Salidas + entradas | Poisson con *offset* de restaurantes 2020, errores robustos | Razón de tasas (*rate ratio*) |

En los dos se comparan dos versiones:

- **Base (quién vive ahí):** tipo de restaurante, municipio, población, escolaridad y competencia.
- **Con generadores (qué atrae gente de día):** lo anterior más el empleo del entorno y la composición de sus establecimientos (escuelas, salud, oficinas, gobierno, comercio grande, ocio).

Si la versión con generadores ajusta mejor, hay evidencia de que lo que atrae gente durante el día importa más allá de quién vive ahí.

Requiere `statsmodels` (`conda install -n tesis-qro -c conda-forge statsmodels`).

## 0. Librerías y rutas

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats

ROOT = Path.cwd().parent
PROC, FIG = ROOT / "data/processed", ROOT / "figures"
RES, MIN_BASE = 8, 5

## 1. Variables explicativas

Todo se mide en **2020**, antes del cambio, y en el **entorno** (hexágono + 6 vecinos), salvo población y escolaridad, que son del hexágono.

| Variable | Cómo se mide | Por qué así |
|---|---|---|
| Población | `log(1 + habitantes)` | Colas largas: pocos hexágonos muy poblados |
| Escolaridad | Grado promedio (15 años y más) | Aproxima el nivel socioeconómico |
| Competencia | % de los establecimientos del entorno que son restaurantes | Saturación de restaurantes |
| Empleo del entorno | `log(1 + empleo estimado)` | Tamaño de la actividad diurna |
| Escuelas, salud, oficinas, gobierno, comercio grande, ocio | % de los establecimientos del entorno | Composición de la actividad |

Los generadores se usan como **porcentaje** y no como conteo porque en el centro todo está junto: con conteos, escuelas, oficinas y salud se mueven casi igual y el modelo no puede separarlas (ver VIF en la sección 2). El porcentaje responde otra pregunta: no cuánto hay, sino **de qué está hecha** la actividad de la zona.

Después todas las variables se **estandarizan** (media 0, desviación 1): cada coeficiente dice cuánto cambia el resultado cuando la variable sube **una desviación estándar**, y los efectos se pueden comparar entre sí.

In [ ]:
modelo = pd.read_parquet(PROC / f"hex_res{RES}_modelo.parquet")
trans = pd.read_parquet(PROC / "transiciones_restaurantes_2020_2025.parquet")

BASE_VARS = ["poblacion", "escolaridad", "competencia"]
GEN_VARS = ["empleo", "escuelas", "salud", "oficinas", "gobierno", "comercio_grande", "ocio"]
ETIQUETAS = {"poblacion": "Población", "escolaridad": "Escolaridad", "competencia": "Competencia (% restaurantes)",
             "empleo": "Empleo del entorno", "escuelas": "Escuelas (%)", "salud": "Salud (%)", "oficinas": "Oficinas (%)",
             "gobierno": "Gobierno (%)", "comercio_grande": "Comercio grande (%)", "ocio": "Ocio y hoteles (%)"}

est = modelo["establecimientos_entorno"]
X = pd.DataFrame({"h3": modelo["h3"], "municipio": modelo["municipio"]})
X["poblacion"] = np.log1p(modelo["POBTOT"].fillna(0))
X["escolaridad"] = modelo["escolaridad"].fillna(modelo["escolaridad"].median())   # hexágonos sin población: mediana
X["competencia"] = modelo["restaurantes_entorno"] / est * 100
X["empleo"] = np.log1p(modelo["empleo_entorno"])
for g in ["escuelas", "salud", "oficinas", "gobierno", "comercio_grande", "ocio"]:
    X[g] = modelo[f"{g}_entorno"] / est * 100
VARS = BASE_VARS + GEN_VARS
X[VARS] = (X[VARS] - X[VARS].mean()) / X[VARS].std()
X.describe().round(2)

## 2. Multicolinealidad

Si dos variables se mueven casi igual (por ejemplo, oficinas y salud en el centro), el modelo no puede separar su efecto y los coeficientes se vuelven inestables. El **VIF** lo mide: arriba de 5 conviene juntar o quitar variables.

In [ ]:
Xv = sm.add_constant(X[VARS])
vif = pd.Series([variance_inflation_factor(Xv.values, i) for i in range(1, Xv.shape[1])], index=VARS, name="VIF")
vif.round(2).sort_values(ascending=False)

## 3. Modelo A · Permanencia de cada restaurante

Logístico: la probabilidad de que un restaurante de 2020 siga en 2025, según su tipo, su municipio y las características de su zona. Los errores estándar se **agrupan por hexágono**, porque los restaurantes de un mismo hexágono comparten entorno y no son observaciones independientes.

In [ ]:
datos_a = trans[["h3", "tipo", "continua"]].merge(X, on="h3", how="left").dropna(subset=VARS + ["municipio"])
datos_a["continua"] = datos_a["continua"].astype(int)
grupos = pd.factorize(datos_a["h3"])[0]

f_base = "continua ~ C(tipo) + C(municipio) + " + " + ".join(BASE_VARS)
f_gen = f_base + " + " + " + ".join(GEN_VARS)
a_base = smf.logit(f_base, datos_a).fit(disp=0, cov_type="cluster", cov_kwds={"groups": grupos})
a_gen = smf.logit(f_gen, datos_a).fit(disp=0, cov_type="cluster", cov_kwds={"groups": grupos})

lr = 2 * (a_gen.llf - a_base.llf)
gl = a_gen.df_model - a_base.df_model
print(f"Restaurantes: {len(datos_a):,} · hexágonos: {datos_a['h3'].nunique():,}")
print(f"Pseudo R² de McFadden → base: {a_base.prsquared:.4f} · con generadores: {a_gen.prsquared:.4f}")
print(f"AIC → base: {a_base.aic:,.1f} · con generadores: {a_gen.aic:,.1f} (menor es mejor)")
print(f"Prueba de razón de verosimilitudes: χ² = {lr:.1f}, gl = {gl:.0f}, p = {stats.chi2.sf(lr, gl):.4f}")

### Lectura del modelo A

Razón de momios por cada desviación estándar: **mayor a 1** = más probabilidad de permanecer; **menor a 1** = menos. Si el intervalo de 95% cruza el 1, no se puede distinguir de "sin efecto".

In [ ]:
def tabla_efectos(res, variables):
    ci = res.conf_int()
    t = pd.DataFrame({"efecto": np.exp(res.params[variables]), "inf_95": np.exp(ci.loc[variables, 0]),
                      "sup_95": np.exp(ci.loc[variables, 1]), "p": res.pvalues[variables]})
    t.index = [ETIQUETAS[v] for v in variables]
    return t

efectos_a = tabla_efectos(a_gen, VARS)
efectos_a.round(3)

In [ ]:
def grafica_efectos(t, titulo, eje, archivo, color):
    t = t.iloc[::-1]
    fig, ax = plt.subplots(figsize=(9, 6))
    y = np.arange(len(t))
    signif = (t["inf_95"] > 1) | (t["sup_95"] < 1)
    ax.hlines(y, t["inf_95"], t["sup_95"], color=np.where(signif, color, "#B8B2A6"), lw=2)
    ax.scatter(t["efecto"], y, color=np.where(signif, color, "#B8B2A6"), zorder=3, s=40)
    ax.axvline(1, color="#666", lw=1, ls="--")
    for yi, (e, s) in enumerate(zip(t["efecto"], signif)):
        ax.text(t["sup_95"].iloc[yi] + 0.01, yi, f"{e:.2f}", va="center", fontsize=9,
                color=color if s else "#8A8478")
    ax.set_yticks(y, t.index); ax.set_xlabel(eje)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_title(titulo, loc="left")
    ax.text(0, -0.12, "En color: intervalo de 95% que no cruza 1. En gris: no se distingue de cero efecto.",
            transform=ax.transAxes, fontsize=9, color="#666")
    plt.tight_layout(); plt.savefig(FIG / archivo, dpi=200, bbox_inches="tight"); plt.show()

grafica_efectos(efectos_a, "Modelo A · Permanencia del restaurante 2020 → 2025",
                "Razón de momios por cada desviación estándar", "05_modelo_permanencia.png", "#14213D")

## 4. Modelo B · Rotación por hexágono

Poisson con *offset*: modela el número de movimientos (salidas + entradas) por cada restaurante que había en 2020, que es exactamente el índice de rotación. Se usan errores robustos (HC1) porque los conteos varían más de lo que supone el Poisson.

Recuerda que las entradas están infladas en la periferia por el Censo Económico 2024: este modelo describe dónde hubo más movimiento registrado, no solo aperturas reales.

In [ ]:
datos_b = modelo[modelo["n_base"] >= MIN_BASE][["h3", "n_base", "salidas", "entradas"]].merge(X, on="h3")
datos_b["movimientos"] = datos_b["salidas"] + datos_b["entradas"]
datos_b = datos_b.dropna(subset=VARS + ["municipio"])

f_base_b = "movimientos ~ C(municipio) + " + " + ".join(BASE_VARS)
f_gen_b = f_base_b + " + " + " + ".join(GEN_VARS)
offset = np.log(datos_b["n_base"])
b_base = smf.glm(f_base_b, datos_b, family=sm.families.Poisson(), offset=offset).fit(cov_type="HC1")
b_gen = smf.glm(f_gen_b, datos_b, family=sm.families.Poisson(), offset=offset).fit(cov_type="HC1")

def devianza_explicada(r):
    return 1 - r.deviance / r.null_deviance

print(f"Hexágonos: {len(datos_b):,}")
print(f"Devianza explicada → base: {devianza_explicada(b_base):.3f} · con generadores: {devianza_explicada(b_gen):.3f}")
print(f"AIC → base: {b_base.aic:,.1f} · con generadores: {b_gen.aic:,.1f}")
efectos_b = tabla_efectos(b_gen, VARS)
efectos_b.round(3)

In [ ]:
grafica_efectos(efectos_b, "Modelo B · Rotación por hexágono 2020 → 2025",
                "Razón de tasas por cada desviación estándar", "05_modelo_rotacion.png", "#A8551A")

## 5. Revisión de residuos por tipo de zona

Si el modelo deja sin explicar algo sistemático del territorio, los residuos se concentran en algún tipo de zona. Lo ideal es que el promedio de cada grupo esté cerca de cero.

In [ ]:
datos_a["residuo"] = a_gen.resid_response
datos_b["residuo"] = b_gen.resid_pearson
zona = modelo.set_index("h3")["tipo_zona"]
pd.DataFrame({
    "Modelo A (prob. observada − predicha)": datos_a.groupby(datos_a["h3"].map(zona))["residuo"].mean(),
    "Modelo B (residuo de Pearson)": datos_b.groupby(datos_b["h3"].map(zona))["residuo"].mean(),
}).round(3)

## 6. Guardar resultados

Se guardan las tablas de efectos y la comparación de ajuste para usarlas en el documento.

In [ ]:
ajuste = pd.DataFrame({
    "Modelo A · base": {"n": len(datos_a), "pseudo_R2": a_base.prsquared, "AIC": a_base.aic},
    "Modelo A · con generadores": {"n": len(datos_a), "pseudo_R2": a_gen.prsquared, "AIC": a_gen.aic},
    "Modelo B · base": {"n": len(datos_b), "pseudo_R2": devianza_explicada(b_base), "AIC": b_base.aic},
    "Modelo B · con generadores": {"n": len(datos_b), "pseudo_R2": devianza_explicada(b_gen), "AIC": b_gen.aic},
}).T
ajuste.to_csv(PROC / "modelo_ajuste.csv", encoding="utf-8-sig")
efectos_a.to_csv(PROC / "modelo_A_permanencia_efectos.csv", encoding="utf-8-sig")
efectos_b.to_csv(PROC / "modelo_B_rotacion_efectos.csv", encoding="utf-8-sig")
ajuste.round(3)

## 7. Cómo leer y cómo no leer estos resultados

- **Son asociaciones, no causas.** Un coeficiente significativo dice que las zonas con más de esa variable tuvieron más (o menos) permanencia, no que poner una escuela haga durar a un restaurante.
- **Un solo periodo.** Todo describe 2020 → 2025, que incluye la pandemia.
- **Entradas infladas.** El modelo B mezcla aperturas reales con negocios que el Censo Económico 2024 apenas registró; el modelo A no tiene ese problema.
- **Casos inciertos.** Los 745 reemplazos inciertos de plazas cuentan como salidas en ambos modelos.
- **Dependencia espacial.** Los hexágonos vecinos se parecen. Los errores agrupados por hexágono (A) y la variable de entorno lo atenúan, pero no lo eliminan.